In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from maskvar.models.vqvae_single import VQVAE_Single
from maskvar.datasets import MaskLevelDatasetDummy
from maskvar.datasets.image_feature_cache import ImageFeatureCache
from maskvar.models.simple_ar import (
    SimpleVAR,
    simple_var_inference,
    simple_var_train_pass
)
from maskvar.maskseg_build_everything import (
    build_hqseg44k_dataset,
    build_simple_var,
    build_vqvae_single_5_stages_v1,
)

device = 'cuda:3'

In [ ]:
# simple_var: SimpleVAR = build_simple_var(simple_var_checkpoint_path='../out/simple_var_1_debug/checkpoints/.simple_var.200.pt', device=device)

simple_var: SimpleVAR = build_simple_var(simple_var_checkpoint_path='../out/ddp_simple_var_coconut_v0_lr2e-3_bs32_sampe_clicks/checkpoints/.simple_var.190848.pt', device=device, enable_prompt_tokens=True)

vqvae: VQVAE_Single = build_vqvae_single_5_stages_v1('../out/out_vqvae_5_stages_v1/ckpt/vqvae_single_epoch_50.pth', require_grad=False)
vqvae = vqvae.to(device)

In [4]:
def visualize(indices, ax, device='cpu', name='mask'):
    result = vqvae.idxBl_to_img(indices, same_shape=True)

    # for i in range(len(indices)):
    #     print(f'index {i}: {indices[i].shape}')
    # result_conv = [edge(item) for item in result]
    result = [mask for mask in result]
    chw = torchvision.utils.make_grid(torch.cat(result, dim=0), nrow=3, padding=1, pad_value=1.0)

    chw = chw.permute(1, 2, 0).cpu().numpy()
    ax.imshow(chw[:, :, 0])
    ax.axis('off')
    ax.set_title(name)

In [5]:
image_feature_cache_train = ImageFeatureCache(
    cache_dir=Path('../data/cache'),
    dataset='hqseg44k_train',
    model_name='sam_vitb',
)

# image_feature_cache_val = ImageFeatureCache(
#     cache_dir=Path('../data/cache'),
#     dataset='hqseg44k_val',
#     model_name='sam_vitb',
# )

train_set, _ = build_hqseg44k_dataset('../data/sam-hq') # validate on train set
train_set_masklevel = MaskLevelDatasetDummy(
    dataset=train_set,
    # sam_encoder=sam_image_encoder,
    image_feature_cache=image_feature_cache_train,
    with_image_embed=True,
    device=device,
    mask_filter_thresh=0.1,
    seed=42,
    count=5,
)
# val_set_masklevel = MaskLevelDatasetDummy(
#     dataset=train_set,
#     # sam_encoder=sam_image_encoder,
#     image_feature_cache=image_feature_cache_val,
#     with_image_embed=True,
#     device=device,
#     mask_filter_thresh=0.1,
#     seed=42,
#     count=5,
# )

train_dataloader = DataLoader(train_set_masklevel, batch_size=1, shuffle=False, drop_last=True)

ImageFeatureCache loaded with metadata:
 {
  "count": 598,
  "resolution": 1024,
  "feature_shape": [
    512,
    256,
    64,
    64
  ],
  "feature_dim": "BCHW",
  "feature_size_in_bytes": 4194304,
  "avg_encode_time": 0.05396735668182373,
  "batch_size": 512,
  "dtype": "torch.float32"
}


Loading DIS5K/DIS-VD: 100%|██████████| 464/464 [00:00<00:00, 363495.90it/s]
/data/clc/maskseg/maskvar/datasets/mask_level_dataset.py:117: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  image_embed_sam = torch.from_numpy(self.image_feature_cache[index])


In [6]:
data_iter = iter(train_dataloader)
# _ = next(data_iter)

In [7]:
image, image_embed_sam, single_mask_normalized, single_mask = next(data_iter)
print("image.shape:", image.shape)
print("image_embed_sam.shape:", image_embed_sam.shape)
print("single_mask_normalized.shape:", single_mask_normalized.shape)
print("single_mask.shape:", single_mask.shape)

image.shape: torch.Size([1, 3, 1024, 1024])
image_embed_sam.shape: torch.Size([1, 256, 64, 64])
single_mask_normalized.shape: torch.Size([1, 1, 256, 256])
single_mask.shape: torch.Size([1, 1, 256, 256])


In [ ]:
# inference (with or without sparse embeddings)
if use_sparse_emb:
    id_seq = simple_var_inference(
        image_feat=image_embed_sam.to(device), 
        simple_var=simple_var, 
        vqvae=vqvae,
        sparse_embeddings=sparse_embeddings
    )
    print("Inference with sparse embeddings")
else:
    id_seq = simple_var_inference(
        image_feat=image_embed_sam.to(device), 
        simple_var=simple_var, 
        vqvae=vqvae
    )
    print("Inference without sparse embeddings")

# gt
idx = vqvae.img_to_idxBl(single_mask_normalized.to(device))

# inference with teacher forced input
if use_sparse_emb:
    logits = simple_var_train_pass(
        idx, 
        image_feat=image_embed_sam.to(device), 
        simple_var=simple_var, 
        vqvae=vqvae,
        sparse_embeddings=sparse_embeddings
    )
    print("Train pass with sparse embeddings")
else:
    logits = simple_var_train_pass(
        idx, 
        image_feat=image_embed_sam.to(device), 
        simple_var=simple_var, 
        vqvae=vqvae
    )
    print("Train pass without sparse embeddings")

# sample the max token
id_seq_teach = logits.argmax(dim=-1)
id_seq_teach_Bl = []
start_pos = 0
for pn in simple_var.patch_num:
    end_pos = start_pos + pn * pn
    id_seq_teach_Bl.append(id_seq_teach[:, start_pos:end_pos])
    start_pos = end_pos

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12,4))

visualize(idx, ax[0], name='gt')
visualize(id_seq, ax[1], name='pred')
visualize(id_seq_teach_Bl, ax[2], name='pred w/ teacher input')

In [ ]:
# Build prompt encoder from SAM (not MobileSAM)
from maskvar.maskseg_build_everything import build_prompt_encoder

try:
    prompt_encoder = build_prompt_encoder('../ckpt/sam_vit_b_01ec64.pth').to(device)
    prompt_encoder.eval()
    
    # Get sparse embeddings from clicks
    with torch.no_grad():
        # Expand batch dimension
        coords_batch = coords.unsqueeze(0)  # (1, N, 2)
        labels_batch = labels.unsqueeze(0)  # (1, N)
        sparse_embeddings, dense_embeddings = prompt_encoder(
            points=(coords_batch, labels_batch),
            boxes=None,
            masks=None
        )
    print(f"Sparse embeddings shape: {sparse_embeddings.shape}")
    use_sparse_emb = True
except Exception as e:
    print(f"Prompt encoder not available: {e}")
    sparse_embeddings = None
    use_sparse_emb = False

In [ ]:
# Visualize clicks on the mask
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(15, 4))

# Original mask
ax[0].imshow(single_mask_np, cmap='gray')
ax[0].set_title('Original Mask')
ax[0].axis('off')

# Distance transform (used for click sampling)
im1 = ax[1].imshow(dt, cmap='hot')
ax[1].set_title('Distance Transform')
ax[1].axis('off')
plt.colorbar(im1, ax=ax[1])

# Mask with clicks overlay
ax[2].imshow(single_mask_np, cmap='gray', alpha=0.7)
for i, (y, x, label) in enumerate(click_list):
    color = 'green' if label == 1 else 'red'
    ax[2].scatter(x, y, c=color, s=100, marker='x', linewidths=2)
    ax[2].annotate(f'{i+1}', (x, y), xytext=(5, 5), textcoords='offset points', 
                   color=color, fontsize=12, fontweight='bold')
ax[2].set_title(f'Mask with {len(click_list)} Clicks')
ax[2].axis('off')

plt.tight_layout()
plt.show()